# 巨大なテキストファイルをメモリに載せずに処理する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/A_colab_python_tips/large_file_stream.ipynb)

NLP クックブック A-14 ／ Colab と Python の小技

数 GB のコーパスを、全部メモリに読み込まずに、1 行ずつ、あるいは一定件数ずつ処理したい。

In [1]:
import gzip
import random

# 動作確認用に 20 万行の gzip 圧縮ファイルを作る
random.seed(0)
words = ["自然", "言語", "処理", "機械", "学習", "翻訳", "モデル", "データ", "評価", "実験"]
with gzip.open("corpus.txt.gz", "wt", encoding="utf-8") as f:
    for i in range(200_000):
        f.write(" ".join(random.choices(words, k=random.randint(3, 12))) + "\n")

In [2]:
from collections import Counter
from itertools import islice

def read_lines(path):
    """ジェネレータ: 1 行ずつ読んで返す。ファイル全体はメモリに載らない"""
    opener = gzip.open if path.endswith(".gz") else open
    with opener(path, "rt", encoding="utf-8") as f:
        for line in f:
            yield line.rstrip("\n")

def batched(iterable, n):
    it = iter(iterable)
    while batch := list(islice(it, n)):
        yield batch

freq, n_lines = Counter(), 0
for batch in batched(read_lines("corpus.txt.gz"), 10_000):    # 1 万行ずつ処理 (モデルの推論ならここでバッチ推論する)
    n_lines += len(batch)
    for line in batch:
        freq.update(line.split())
print(n_lines, "行", freq.most_common(3))

200000 行 [('モデル', 150360), ('言語', 150293), ('学習', 150243)]


In [3]:
# ファイル全体を読まずに一様ランダムに k 行を選ぶ (リザーバサンプリング)
def reservoir_sample(iterable, k, seed=0):
    rng, sample = random.Random(seed), []
    for i, item in enumerate(iterable):
        if i < k:
            sample.append(item)
        elif (j := rng.randint(0, i)) < k:
            sample[j] = item
    return sample

print(reservoir_sample(read_lines("corpus.txt.gz"), 3))

['モデル データ データ 実験 学習', '学習 処理 翻訳 言語 評価 翻訳 学習 機械 実験 学習 データ 翻訳', '翻訳 処理 自然 モデル 自然 処理']


- `f.readlines()` や `f.read().split("\n")` は全体をメモリに載せるので、大きなファイルでは使わない。
- 行数だけ知りたいなら `!zcat corpus.txt.gz | wc -l`、先頭を見たいなら `!zcat corpus.txt.gz | head` が速い。
- 表形式なら `pd.read_csv(path, chunksize=100_000)`、Hugging Face Datasets なら `streaming=True` が同じ役割をする。